# Example: Using the Utils Module

This notebook demonstrates how to use the modular utils framework to:
- Easily swap return generation methods
- Test different allocation policies
- Compare spending strategies
- Mix and match components

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import sys
sys.path.append('..')  # Add Code directory to path

from utils import (
    # Return generators
    CholeskyBootstrapReturns,
    BlockBootstrapReturns,
    
    # Allocation policies
    ConstantAllocation,
    TimeBasedPolicy,
    ControlMatrixPolicy,
    
    # Spending policies
    PercentageOfWealth,
    FloorCeilingSpending,
    PensionPlusPercentage,
    
    # Simulation
    simulate_wealth_trajectory,
    
    # Objectives
    log_consumption_utility,
    smoothness_penalty,
    
    # Helpers
    load_historical_returns,
    sample_without_replacement
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## 1. Load Historical Data

In [ ]:
# Load returns
try:
    yearly_returns, mean_returns, cov_matrix = load_historical_returns(source="sallypy")
except:
    yearly_returns, mean_returns, cov_matrix = load_historical_returns(
        source="csv",
        csv_path="WIP/returns.csv"
    )

print("Mean returns:")
print(mean_returns)
print("\nCovariance matrix:")
print(cov_matrix)

## 2. Compare Return Generators

In [ ]:
# Option 1: Cholesky decomposition (parametric)
cholesky_gen = CholeskyBootstrapReturns(mean_returns, cov_matrix)
cholesky_returns = cholesky_gen.generate(n_simulations=1000, n_timesteps=40)

# Option 2: Block bootstrap (non-parametric)
block_gen = BlockBootstrapReturns(yearly_returns, block_size=5)
block_returns = block_gen.generate(n_simulations=1000, n_timesteps=40)

# Compare distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cholesky_returns[:, :, 1].flatten(), bins=50, alpha=0.6, label='Cholesky')
axes[0].hist(block_returns[:, :, 1].flatten(), bins=50, alpha=0.6, label='Block Bootstrap')
axes[0].set_title('Stock Returns Distribution')
axes[0].set_xlabel('Return')
axes[0].legend()

axes[1].scatter(
    cholesky_returns[:, :, 0].flatten(),
    cholesky_returns[:, :, 1].flatten(),
    alpha=0.1, s=10, label='Cholesky'
)
axes[1].scatter(
    block_returns[:, :, 0].flatten(),
    block_returns[:, :, 1].flatten(),
    alpha=0.1, s=10, label='Block Bootstrap'
)
axes[1].set_title('Bond vs Stock Returns')
axes[1].set_xlabel('Bond Return')
axes[1].set_ylabel('Stock Return')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Test Different Allocation Policies

In [ ]:
# Simulation parameters
N_SIMS = 5000
N_YEARS = 40
INITIAL_WEALTH = 500000

# Generate returns (using Cholesky for this example)
returns_np = cholesky_gen.generate(N_SIMS, N_YEARS)
returns_torch = torch.tensor(returns_np, dtype=torch.float32, device=DEVICE)

# Define different allocation policies to compare
policies = {
    '100% Stocks': ConstantAllocation(1.0),
    '60/40': ConstantAllocation(0.6),
    'Age-based (100-age)': TimeBasedPolicy(
        policy_nodes=torch.tensor([1.0, 0.5, 0.2], device=DEVICE),
        n_timesteps=N_YEARS
    ),
}

# Use simple spending policy for comparison
spending = FloorCeilingSpending(
    rate=0.04,
    floor_real=30000,
    inflation=0.03,
    real_decline_rate=0.02
)

# Run simulations
results = {}
for name, policy in policies.items():
    wealth, consumption = simulate_wealth_trajectory(
        returns=returns_torch,
        allocation_policy=policy,
        spending_policy=spending,
        initial_wealth=INITIAL_WEALTH
    )
    results[name] = {
        'wealth': wealth.cpu().numpy(),
        'consumption': consumption.cpu().numpy()
    }

print("Simulations complete!")

In [ ]:
# Compare terminal wealth distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Set2(range(len(policies)))

# Terminal wealth
for (name, data), color in zip(results.items(), colors):
    terminal = data['wealth'][:, -1]
    axes[0].hist(terminal / 1000, bins=50, alpha=0.5, label=name, color=color)

axes[0].set_xlabel('Terminal Wealth ($1000s)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Terminal Wealth Distribution')
axes[0].legend()

# Wealth percentiles over time
ages = np.arange(65, 65 + N_YEARS + 1)

for (name, data), color in zip(results.items(), colors):
    median_wealth = np.percentile(data['wealth'], 50, axis=0)
    axes[1].plot(ages, median_wealth / 1000, label=name, linewidth=2, color=color)

axes[1].set_xlabel('Age')
axes[1].set_ylabel('Median Wealth ($1000s)')
axes[1].set_title('Median Wealth Trajectory')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Compare Spending Policies

In [ ]:
# Use constant 60/40 allocation for comparison
allocation = ConstantAllocation(0.6)

# Different spending approaches
spending_policies = {
    '4% Rule': PercentageOfWealth(0.04),
    
    'Floor + 3%': FloorCeilingSpending(
        rate=0.03,
        floor_real=30000,
        inflation=0.03,
        real_decline_rate=0.02
    ),
    
    'NZ Super + 2%': PensionPlusPercentage(
        pension_real=27456,  # NZ Super for single person 2024
        wealth_rate=0.02,
        inflation=0.03
    ),
}

# Run simulations
spending_results = {}
for name, spend_policy in spending_policies.items():
    wealth, consumption = simulate_wealth_trajectory(
        returns=returns_torch,
        allocation_policy=allocation,
        spending_policy=spend_policy,
        initial_wealth=INITIAL_WEALTH
    )
    spending_results[name] = {
        'wealth': wealth.cpu().numpy(),
        'consumption': consumption.cpu().numpy()
    }

print("Spending policy comparison complete!")

In [ ]:
# Compare consumption patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ages = np.arange(65, 65 + N_YEARS)
colors = plt.cm.Set1(range(len(spending_policies)))

for (name, data), color in zip(spending_results.items(), colors):
    consumption = data['consumption']
    wealth = data['wealth']
    
    # Median consumption
    median_cons = np.percentile(consumption, 50, axis=0)
    axes[0, 0].plot(ages, median_cons / 1000, label=name, linewidth=2, color=color)
    
    # Consumption distribution at age 85 (year 20)
    axes[0, 1].hist(consumption[:, 20] / 1000, bins=30, alpha=0.5, label=name, color=color)
    
    # Median wealth
    median_wealth = np.percentile(wealth, 50, axis=0)
    axes[1, 0].plot(ages, median_wealth[:-1] / 1000, label=name, linewidth=2, color=color)
    
    # Bankruptcy rate over time
    bankruptcy_rate = (wealth == 0).sum(axis=0) / N_SIMS * 100
    axes[1, 1].plot(ages, bankruptcy_rate[:-1], label=name, linewidth=2, color=color)

axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Median Consumption ($1000s)')
axes[0, 0].set_title('Consumption Over Time')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].set_xlabel('Consumption at Age 85 ($1000s)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Consumption Distribution (Age 85)')
axes[0, 1].legend()

axes[1, 0].set_xlabel('Age')
axes[1, 0].set_ylabel('Median Wealth ($1000s)')
axes[1, 0].set_title('Wealth Over Time')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].set_xlabel('Age')
axes[1, 1].set_ylabel('Bankruptcy Rate (%)')
axes[1, 1].set_title('Bankruptcy Rate Over Time')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Example: Optimize a Time-Based Policy

Quick example showing optimization with the modular framework.

In [ ]:
# Initialize policy nodes
policy_nodes = torch.tensor(
    [0.9, 0.7, 0.5, 0.3],
    requires_grad=True,
    device=DEVICE
)

# Spending policy
spending_policy = FloorCeilingSpending(
    rate=0.04,
    floor_real=30000,
    inflation=0.03,
    real_decline_rate=0.02
)

# Optimizer
optimizer = torch.optim.Adam([policy_nodes], lr=0.01)

# Training loop
N_ITERATIONS = 200
cost_history = []

print("Optimizing time-based policy...\n")

for iteration in range(N_ITERATIONS):
    optimizer.zero_grad()
    
    # Create policy with current nodes
    allocation_policy = TimeBasedPolicy(policy_nodes, N_YEARS)
    
    # Sample returns
    batch_returns = sample_without_replacement(returns_torch, 1000)
    
    # Simulate
    wealth, consumption = simulate_wealth_trajectory(
        returns=batch_returns,
        allocation_policy=allocation_policy,
        spending_policy=spending_policy,
        initial_wealth=INITIAL_WEALTH
    )
    
    # Calculate cost
    cost = log_consumption_utility(consumption)
    cost += smoothness_penalty(policy_nodes, weight=0.01)
    
    # Backprop
    cost.backward()
    optimizer.step()
    
    # Clamp to [0, 1]
    with torch.no_grad():
        policy_nodes.clamp_(0.0, 1.0)
    
    cost_history.append(cost.item())
    
    if (iteration + 1) % 50 == 0:
        print(f"Iteration {iteration + 1}: Cost = {cost.item():.6f}")
        print(f"  Nodes: {policy_nodes.detach().cpu().numpy()}")

print("\nOptimization complete!")
print(f"Optimal nodes: {policy_nodes.detach().cpu().numpy()}")

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cost_history)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost')
axes[0].set_title('Optimization Progress')
axes[0].grid(True, alpha=0.3)

# Show optimal allocation path
with torch.no_grad():
    optimal_policy = TimeBasedPolicy(policy_nodes, N_YEARS)
    time_points = torch.arange(N_YEARS, dtype=torch.float32, device=DEVICE)
    dummy_wealth = torch.ones(N_YEARS, device=DEVICE)
    allocations = optimal_policy.get_allocation(time_points, dummy_wealth).cpu().numpy()

axes[1].plot(ages, allocations * 100, linewidth=2)
axes[1].scatter(
    65 + np.linspace(0, N_YEARS - 1, len(policy_nodes)),
    policy_nodes.detach().cpu().numpy() * 100,
    s=100, c='red', zorder=5, label='Policy Nodes'
)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Stock Allocation (%)')
axes[1].set_title('Optimal Time-Based Allocation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

The utils module provides:

### Return Generators
- `CholeskyBootstrapReturns`: Parametric (uses mean and covariance)
- `BlockBootstrapReturns`: Non-parametric (preserves time dependencies)

### Allocation Policies
- `ConstantAllocation`: Fixed allocation (e.g., 60/40)
- `TimeBasedPolicy`: Age-based with interpolated nodes
- `ControlMatrixPolicy`: 2D time × wealth control matrix

### Spending Policies
- `PercentageOfWealth`: Simple percentage rule
- `FloorCeilingSpending`: Floor with optional ceiling
- `PensionPlusPercentage`: Fixed pension + variable spending

All components are designed to work together seamlessly using the `simulate_wealth_trajectory` function!